Vấn đề 1:

In [39]:
import pandas as pd
import numpy as np

column_names = ["Id", "Name", "Age", "Weight", "m0006", "m0612", "m1218", "f0006", "f0612", "f1218", "Extra_Column"]

df = pd.read_csv("patient_heart_rate.csv", names=column_names)

Vấn đề 2:

In [40]:
df[['Firstname', 'Lastname']] = df['Name'].str.split(' ', n=1, expand=True)
df.drop('Name', axis=1, inplace=True)

Vấn đề 3:

In [41]:
def convert_weight(x):
    x = str(x)
    if "lbs" in x:
        val = float(x.replace("lbs", ""))
        return str(int(val/2.2)) + "kgs"
    elif "kgs" in x:
        return x
    else:
        return np.nan

df['Weight'] = df['Weight'].apply(convert_weight)

Vấn đề 4:

In [42]:
df.dropna(how="all", inplace=True)

Vấn đề 5:

In [43]:
df.drop_duplicates(subset=['Firstname','Lastname','Age','Weight'], inplace=True)

Vấn đề 6:

In [44]:
df['Firstname'] = df['Firstname'].replace({r'[^\x00-\x7F]+':''}, regex=True)
df['Lastname'] = df['Lastname'].replace({r'[^\x00-\x7F]+':''}, regex=True)

Vấn đề 7:

In [45]:
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Weight'] = df['Weight'].replace("nan", np.nan)
df['Weight'] = df['Weight'].fillna(df['Weight'].mode()[0])

Vấn đề 8:

In [46]:
df_melt = pd.melt(df, id_vars=['Id','Age','Weight','Firstname','Lastname'],
                  value_name="PulseRate", var_name="sex_and_time")

tmp_df = df_melt["sex_and_time"].str.extract(r"([mf])(\d{2})(\d{2})")
tmp_df.columns = ["Sex", "HourLower", "HourUpper"]
tmp_df["Time"] = tmp_df["HourLower"] + "-" + tmp_df["HourUpper"]

df_final = pd.concat([df_melt.drop('sex_and_time', axis=1), tmp_df], axis=1)

Câu 11:

In [47]:
df_final['PulseRate'] = df_final['PulseRate'].replace("-", np.nan)
df_final['PulseRate'] = pd.to_numeric(df_final['PulseRate'])
df_final['PulseRate'] = df_final.groupby('Id')['PulseRate'].transform(
    lambda x: x.fillna(x.mean())
)

Câu 12:

In [48]:
df_final.to_csv("patient_heart_rate_clean.csv", index=False)

print(df_final.head())

    Id   Age Weight Firstname Lastname  PulseRate Sex HourLower HourUpper  \
0  1.0  56.0  70kgs     Micky     Mous  72.000000   m        00        06   
1  2.0  34.0  70kgs    Donald     Duck  81.666667   m        00        06   
2  3.0  16.0  70kgs      Mini    Mouse  68.666667   m        00        06   
3  4.0  36.1  78kgs   Scrooge   McDuck  78.000000   m        00        06   
4  5.0  54.0  90kgs      Pink  Panther  72.000000   m        00        06   

    Time  
0  00-06  
1  00-06  
2  00-06  
3  00-06  
4  00-06  
